# Kiểm tra hai nguồn vector trên cùng slot YOLO
Chọn CPU, bật Internet, thêm secret codex có quyền đọc v135. Gắn csa-scst-full-karpathy-cache-v1 và csa-prompt-full-karpathy-v175. Gửi result.json.


In [ ]:
EXPECTED_RESULT='d8fa2d6e0dd2af871aef87e7cf53a093a98f1d4ede3962b895c8f4f397743784'
EXPECTED_FEATURES='a72cc7e4daf5024ada23cb47fba90be997693ab6cd42be8ae223ae4cf9b95030'
PARTS=[{'part': 0, 'receipt_name': 'scst_cache_receipt.json', 'receipt_sha256': '3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510', 'archive_sha256': '64eb7d3c958fca4996d717c300191022176e5d688ddd4ba6ea94a5f1a30d3946', 'train_start': 0, 'train_stop': 40000, 'ids_sha256': 'f1fd1507bb3dc6b5dd4ddc57a83e723e2610ec7a45e7a6b05813fa633e519480', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/82'}, {'part': 1, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '7ba08f77f38ce48fc8a8a023dbe70ffd6b9bed142f28d54711a9c3ebd0b98a12', 'archive_sha256': 'b9cadfb1f55067b767ed1e32012d5b5bfc7f9f0000c9ff8df0c2ed9676ab461c', 'train_start': 40000, 'train_stop': 80000, 'ids_sha256': '570fab90751d77fda57aebe84a17537a7cba57d3244fc06a5cdcb55ff2c1d885', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/83'}, {'part': 2, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '3b8247a0e3fc24aff0a0b2125ce5fbd372976274c5fcf5e1f46315516824ad5c', 'archive_sha256': 'a78982bb44fbb4ce5668d36f9df47de5da9536834ac379f123349a893fd0c9c8', 'train_start': 80000, 'train_stop': 113287, 'ids_sha256': '22d47e2713dbe61777a97a8104ba4bc2490e253c4ff8fd71d0e871d57b2d38fb', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/84'}]

from pathlib import Path
import hashlib, io, json, os, subprocess, sys, tempfile, zipfile
import numpy as np

VERSION='ducanh3/notebook7bc83c4c76/135'
SECRET_NAME='codex'
MOUNTED_V135=''  # Optional mounted folder containing the two exact output files.
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
CLIP_ROOT=Path('/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1')
OUT=Path('/kaggle/working/regionclip_slot_pair160')
temp=Path(tempfile.mkdtemp())
if MOUNTED_V135:
    root=Path(MOUNTED_V135)
else:
    from kaggle_secrets import UserSecretsClient
    cli=temp/'cli'
    subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(cli),'kaggle==2.2.5'],
                   check=True,timeout=300)
    env=dict(os.environ)
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
        env.pop(key,None)
    env['KAGGLE_API_TOKEN']=UserSecretsClient().get_secret(SECRET_NAME).strip()
    env['PYTHONPATH']=str(cli)
    root=temp/'download'
    root.mkdir()
    api=[sys.executable,'-c','from kaggle.cli import main; main()', 'kernels','output',
         VERSION,'-p',str(root),'--file-pattern',r'(^|/)regionclip_slot_cache160/(result[.]json|region_features[.]npz)$']
    subprocess.run(api,env=env,check=True,timeout=300)

def sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

result_path,=(path for path in root.rglob('result.json') if sha(path)==EXPECTED_RESULT)
feature_path=result_path.with_name('region_features.npz')
if sha(feature_path)!=EXPECTED_FEATURES:
    raise ValueError('RegionCLIP NPZ SHA differs from audited v135')
result=json.loads(result_path.read_bytes())
source={row['coco_id']:row for row in result['records']}
if len(source)!=160:
    raise ValueError('Wrong v135 cohort')
link_path,=PROMPT_ROOT.rglob('prompt_links.jsonl')
if sha(link_path)!=result['prompt_links_sha256']:
    raise ValueError('Wrong v175 links')
with link_path.open(encoding='utf8') as stream:
    links={r['coco_id']:r for r in map(json.loads,stream) if r['coco_id'] in source}
if set(links)!=set(source):
    raise ValueError('Missing v175 links')

def box_weights(value):
    b=np.asarray(value,np.float64)
    if b.shape!=(4,) or not np.isfinite(b).all() or (b<0).any() or (b>1).any() or (b[2:]<=b[:2]).any():raise ValueError('Invalid normalized xyxy box')
    x0,y0,x1,y1=b*14;axis=np.arange(14,dtype=np.float64)
    wx=np.maximum(0,np.minimum(axis+1,x1)-np.maximum(axis,x0));wy=np.maximum(0,np.minimum(axis+1,y1)-np.maximum(axis,y0));weights=(wy[:,None]*wx[None,:]).reshape(196)
    if weights.sum()<=0:raise ValueError('Box has no patch support')
    return (weights/weights.sum()).astype(np.float32)

clip={}
with zipfile.ZipFile(CLIP_ROOT/'cache_metadata_zip.bin') as metadata:
    for part in PARTS:
        idx=part['part']
        raw=metadata.read(f"part{idx}/{part['receipt_name']}")
        if hashlib.sha256(raw).hexdigest()!=part['receipt_sha256']:
            raise ValueError('Unexpected CLIP cache receipt')
        receipt=json.loads(raw)['files_sha256']
        groups=[('train','pilot_ids.npy','pilot_patches.npy')]
        if idx==0:
            groups.append(('valid','ids.npy','patches.npy'))
        for directory,ids_name,features_name in groups:
            member=f'{directory}/{ids_name}'
            raw_ids=metadata.read(f'part{idx}/{member}')
            if hashlib.sha256(raw_ids).hexdigest()!=receipt[member]:
                raise ValueError('Unexpected CLIP cache IDs')
            ids=np.load(io.BytesIO(raw_ids),allow_pickle=False)
            patches=np.load(CLIP_ROOT/f'part{idx}_{directory}_{features_name}',mmap_mode='r',allow_pickle=False)
            if patches.shape!=(len(ids),196,768):
                raise ValueError('Unexpected CLIP patch array shape')
            for index,image_id in enumerate(ids):
                cid=int(image_id)
                if cid not in source:
                    continue
                expected='train' if directory=='train' else 'val'
                if cid in clip or source[cid]['group']!=expected:
                    raise ValueError('Duplicate or wrong-split CLIP row')
                clip[cid]=np.array(patches[index],dtype=np.float32,copy=True)
if set(clip)!=set(source):
    raise ValueError('Incomplete CLIP rows')
ids=sorted(source)
region=np.zeros((160,8,1024),np.float32)
pooled=np.zeros((160,8,768),np.float32)
boxes=np.zeros((160,8,4),np.float32)
mask=np.zeros((160,8),bool)
with np.load(feature_path,allow_pickle=False) as vectors:
    if set(vectors.files)!={str(cid) for cid in ids}:
        raise ValueError('Unexpected RegionCLIP image IDs')
    for i,cid in enumerate(ids):
        record=source[cid]
        row=links[cid]
        slots=[dict(instance_id=o['instance_id'],roi=o['roi'],bbox=o['roi_bbox'],text=o['text']) for o in row['objects']]
        if slots!=record['slots'] or row['group']!=record['group'] or row['filename']!=record['filename']:
            raise ValueError(f'Changed slot alignment for {cid}')
        count=len(slots)
        vector=vectors[str(cid)]
        if vector.shape!=(count,1024) or vector.dtype!=np.float32:
            raise ValueError(f'Unexpected RegionCLIP vector for {cid}')
        coords=[s['bbox'] for s in slots]
        pooled[i,:count]=np.stack([box_weights(b) for b in coords])@clip[cid]
        region[i,:count]=vector
        boxes[i,:count]=coords
        mask[i,:count]=True
if int(mask.sum())!=523 or not all(np.isfinite(a).all() for a in (pooled,region)):
    raise ValueError('Invalid paired vectors')
if not all((np.linalg.norm(a[mask],axis=-1)>0).all() for a in (pooled,region)):
    raise ValueError('Zero active vector')
OUT.mkdir(exist_ok=True)
path=OUT/'paired_slots.npz'
np.savez(path,coco_ids=ids,clip=pooled,regionclip=region,boxes=boxes,valid=mask)
summary=dict(source=VERSION,images=len(ids),slots=int(mask.sum()),group_counts=result['images'],
             clip_shape=list(pooled.shape),regionclip_shape=list(region.shape),paired_sha256=sha(path),
             full_clip_patch_hash_checked=False)
(OUT/'result.json').write_text(json.dumps(summary,indent=2),encoding='utf8')
print(summary)
print('SEND:',OUT/'result.json')
